Train a Perceiver-style compressor that maps T5 encoder hidden states from 256 tokens to 128 tokens while preserving frozen decoder outputs.

In [ ]:
!pip install transformers datasets accelerate sentencepiece

In [ ]:
!pip uninstall -y transformers datasets huggingface_hub
!pip install transformers==4.52.4 datasets==3.6.0 huggingface_hub==0.33.5

Found existing installation: transformers 5.9.0
Uninstalling transformers-5.9.0:
  Successfully uninstalled transformers-5.9.0
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: huggingface_hub 1.17.0
Uninstalling huggingface_hub-1.17.0:
  Successfully uninstalled huggingface_hub-1.17.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.7/515.7 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 95.4 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [ ]:
import transformers
import datasets
import huggingface_hub

print(transformers.__version__)
print(datasets.__version__)
print(huggingface_hub.__version__)

4.52.4
3.6.0
0.33.5


In [ ]:
from datasets import load_dataset

dataset = load_dataset("ag_news")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

from transformers.modeling_outputs import (
    BaseModelOutput,
)

In [ ]:
MODEL_NAME = "google/flan-t5-base"

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
).to(device)

model.eval()

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [ ]:
for p in model.parameters():
    p.requires_grad = False

In [ ]:
class PerceiverCompressor(
    nn.Module
):
    def __init__(
        self,
        hidden_size=768,
        num_latents=128,
        heads=8,
    ):
        super().__init__()

        self.latents = nn.Parameter(
            torch.randn(
                num_latents,
                hidden_size
            )
        )

        self.cross_attn = (
            nn.MultiheadAttention(
                hidden_size,
                heads,
                batch_first=True,
            )
        )

        self.norm = nn.LayerNorm(
            hidden_size
        )

    def forward(
        self,
        hidden_states,
    ):
        B = hidden_states.size(0)

        latents = (
            self.latents
            .unsqueeze(0)
            .expand(B,-1,-1)
        )

        compressed,_ = (
            self.cross_attn(
                latents,
                hidden_states,
                hidden_states,
            )
        )

        return self.norm(
            compressed
        )

In [ ]:
compressor = (
    PerceiverCompressor()
    .to(device)
)

optimizer = torch.optim.AdamW(
    compressor.parameters(),
    lr=1e-4,
)

In [ ]:
dataset = load_dataset(
      "ag_news"
)

train_texts = []

for x in dataset["train"]:
    txt = x["text"]

    if len(txt) > 20:
        train_texts.append(txt)

print(
    len(train_texts)
)

120000


In [ ]:
def training_step(text):

    inputs = tokenizer(
        text,
        max_length=256,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        enc = model.encoder(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            return_dict=True,
        )

    hidden = enc.last_hidden_state

    compressed = compressor(
        hidden
    )
    if torch.rand(1).item() < 0.0005:
      print(
        "hidden:",
        hidden.shape,
        "compressed:",
        compressed.shape,
    )


    decoder_input_ids = model._shift_right(
        inputs["input_ids"]
    )

    with torch.no_grad():

        teacher_outputs = model.decoder(
            input_ids=decoder_input_ids,
            encoder_hidden_states=hidden,
            encoder_attention_mask=inputs[
                "attention_mask"
            ],
            output_hidden_states=True,
            return_dict=True,
        )

    student_outputs = model.decoder(
        input_ids=decoder_input_ids,
        encoder_hidden_states=compressed,
        encoder_attention_mask=torch.ones(
            compressed.shape[:2],
            device=device,
            dtype=torch.long,
        ),
        output_hidden_states=True,
        return_dict=True,
    )

    teacher_hidden = (
        teacher_outputs.last_hidden_state
    )

    student_hidden = (
        student_outputs.last_hidden_state
    )

    loss = F.mse_loss(
        student_hidden,
        teacher_hidden,
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    return loss.item()

In [12]:
for epoch in range(5):

    total = 0

    for i,text in enumerate(
        train_texts[:5000]
    ):

        loss = training_step(
            text
        )

        total += loss

        if i % 100 == 0:

            print(
                epoch,
                i,
                total/(i+1)
            )

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


0 0 0.16234935820102692
0 100 0.047961232648922665
0 200 0.045177656897709736
0 300 0.04267511727207919
0 400 0.03921468447698768
0 500 0.03839148522285882
0 600 0.03776359210370384
0 700 0.03719658757720711
0 800 0.03640850262481547
0 900 0.03595792912939082
0 1000 0.03588419245717885
0 1100 0.03575138446894372
0 1200 0.035518798133440554
0 1300 0.03527136670739077
0 1400 0.035063383518468896
0 1500 0.034861884172323304
0 1600 0.034548376163329314
0 1700 0.0343650453628152
0 1800 0.03422305885565854
0 1900 0.034046672491835864
0 2000 0.03395326025569918
0 2100 0.033768577983905194
0 2200 0.033554432719018396
0 2300 0.0333103064491354
0 2400 0.03303556213087896
hidden: torch.Size([1, 256, 768]) compressed: torch.Size([1, 128, 768])
0 2500 0.03261285036003909
0 2600 0.032222728596578445
0 2700 0.031814540981879524
0 2800 0.03144798003454978
0 2900 0.031117375674588502
0 3000 0.030770213328927546
0 3100 0.030430121514892165
0 3200 0.03011049578063509
0 3300 0.029816664434276507
0 3400 0.

KeyboardInterrupt: 

In [13]:
torch.save(
    compressor.state_dict(),
    "compressor_256_to_128.pt"
)

In [14]:
from google.colab import files

files.download(
    "compressor_256_to_128.pt"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>